In [3]:
%pip install pandas 

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.8 MB 11.4 MB/s eta 0:00:01
   -------------------- ------------------- 5.0/9.8 MB 12.9 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.8 MB 12.9 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.8 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.8 MB 10.6 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 8.9 MB/s  0:00:01
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   --------- ------------------------------ 2.9/12.4 MB 12.7 MB/s eta 0:00:01
   ----------------- ---------------------- 5.5/12.4 MB 13.3 MB/s eta 0:00:01
   ------------------------- -------------- 7.9/12.4 MB 12.2 MB/s eta 0:00:01
   ---------------------------------- ----- 10.7/12.4 MB 12.4 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 11.5 MB/s  0:00:01

   ----------

In [5]:
%pip install sqlalchemy pg8000

  Using cached sqlalchemy-2.0.51-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached pg8000-1.31.5-py3-none-any.whl.metadata (88 kB)
  Using cached greenlet-3.5.3-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached asn1crypto-1.5.1-py2.py3-none-any.whl.metadata (13 kB)
Using cached sqlalchemy-2.0.51-cp313-cp313-win_amd64.whl (2.1 MB)
Using cached pg8000-1.31.5-py3-none-any.whl (57 kB)
Using cached greenlet-3.5.3-cp313-cp313-win_amd64.whl (239 kB)
Using cached asn1crypto-1.5.1-py2.py3-none-any.whl (105 kB)

   ---------------------------------------- 0/5 [asn1crypto]
   -------- ------------------------------- 1/5 [scramp]
   ---------------- ----------------------- 2/5 [greenlet]
   ---------------- ----------------------- 2/5 [greenlet]
   ------------------------ --------------- 3/5 [sqlalchemy]
   ------------------------ --------------- 3/5 [sqlalchemy]
   ------------------------ --------------- 3/5 [sqlalchemy]
   ------------------------ --------------- 3/5 [sqlalc

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

path = "C:/Users/lukin/OneDrive/Documentos/projeto olist/"

print("Carregando arquivos CSV...")

customers = pd.read_csv(f"{path}olist_customers_dataset.csv")
orders = pd.read_csv(f"{path}olist_orders_dataset.csv")
order_items = pd.read_csv(f"{path}olist_order_items_dataset.csv")
order_payments = pd.read_csv(f"{path}olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(f"{path}olist_order_reviews_dataset.csv")
products = pd.read_csv(f"{path}olist_products_dataset.csv")
sellers = pd.read_csv(f"{path}olist_sellers_dataset.csv")
category_translation = pd.read_csv(f"{path}product_category_name_translation.csv")

print("Todos os arquivos foram carregados com sucesso!")

cols_datetime = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in cols_datetime:
    orders[col] = pd.to_datetime(orders[col])

orders['dias_entrega_real'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['dias_atraso'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
orders['foi_atrasado'] = orders['dias_atraso'].apply(lambda x: 1 if x > 0 else 0)

products = products.merge(category_translation, on='product_category_name', how='left')

print("\n--- RESUMO DOS DADOS ---")
print(f"Total de pedidos analisados: {len(orders):,}")
print(f"Total de clientes únicos: {customers['customer_unique_id'].nunique():,}")
print(f"Taxa de atraso nas entregas: {orders['foi_atrasado'].mean() * 100:.2f}%\n")

USUARIO = "postgres"
SENHA = "Postgres1312"
HOST = "localhost"
PORTA = "5432"
BANCO = "olist_db"

print("Verificando/Criando banco de dados no PostgreSQL...")

engine_init = create_engine(
    f'postgresql+pg8000://{USUARIO}:{SENHA}@{HOST}:{PORTA}/postgres',
    isolation_level="AUTOCOMMIT"
)

with engine_init.connect() as conn:
    res = conn.execute(text(f"SELECT 1 FROM pg_database WHERE datname='{BANCO}'"))
    if not res.scalar():
        conn.execute(text(f"CREATE DATABASE {BANCO}"))
        print(f"Banco de dados '{BANCO}' criado com sucesso!")
    else:
        print(f"Banco de dados '{BANCO}' já existe.")

print("Exportando tabelas para o PostgreSQL...")

engine = create_engine(f'postgresql+pg8000://{USUARIO}:{SENHA}@{HOST}:{PORTA}/{BANCO}')

customers.to_sql('raw_customers', engine, if_exists='replace', index=False)
orders.to_sql('raw_orders', engine, if_exists='replace', index=False)
order_items.to_sql('raw_order_items', engine, if_exists='replace', index=False)
order_payments.to_sql('raw_order_payments', engine, if_exists='replace', index=False)
order_reviews.to_sql('raw_order_reviews', engine, if_exists='replace', index=False)
products.to_sql('raw_products', engine, if_exists='replace', index=False)
sellers.to_sql('raw_sellers', engine, if_exists='replace', index=False)

print("SUCESSO! Todas as 7 tabelas foram exportadas para o PostgreSQL!")

Carregando arquivos CSV...
Todos os arquivos foram carregados com sucesso!

--- RESUMO DOS DADOS ---
Total de pedidos analisados: 99,441
Total de clientes únicos: 96,096
Taxa de atraso nas entregas: 6.57%

Verificando/Criando banco de dados no PostgreSQL...
Banco de dados 'olist_db' já existe.
Exportando tabelas para o PostgreSQL...
SUCESSO! Todas as 7 tabelas foram exportadas para o PostgreSQL!
